In [1]:
print("PayGuard AI Risk Manager")
print("Environment is ready!")

PayGuard AI Risk Manager
Environment is ready!


In [2]:
!pip install -q pandas numpy scikit-learn matplotlib gradio joblib

In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Number of simulated transactions
n = 10000

# Generate transaction features
amount = np.random.lognormal(mean=7.0, sigma=1.0, size=n)
amount = np.clip(amount, 50, 100000)

account_age_days = np.random.randint(1, 2000, n)

transactions_last_24h = np.random.poisson(3, n)
transactions_last_24h = np.clip(transactions_last_24h, 0, 30)

device_changes_30d = np.random.poisson(1, n)
device_changes_30d = np.clip(device_changes_30d, 0, 10)

failed_attempts = np.random.poisson(0.8, n)
failed_attempts = np.clip(failed_attempts, 0, 10)

international = np.random.binomial(1, 0.18, n)

new_device = np.random.binomial(1, 0.20, n)

# Transaction hour
transaction_hour = np.random.randint(0, 24, n)

# Calculate a risk score used to simulate fraud patterns
risk_score = (
    (amount > 50000) * 2.0 +
    (transactions_last_24h > 8) * 2.0 +
    (device_changes_30d > 3) * 1.5 +
    (failed_attempts > 2) * 1.5 +
    (international == 1) * 0.8 +
    (new_device == 1) * 1.5 +
    ((transaction_hour < 5) | (transaction_hour > 23)) * 1.2 +
    (account_age_days < 30) * 1.2
)

# Convert risk into probability of fraud
fraud_probability = 1 / (1 + np.exp(-(risk_score - 3.5)))

# Add randomness so the model has a genuine prediction problem
fraud_probability = np.clip(
    fraud_probability * 0.75 + np.random.uniform(0, 0.15, n),
    0,
    0.95
)

is_fraud = np.random.binomial(1, fraud_probability)

# Create DataFrame
df = pd.DataFrame({
    "amount": amount.round(2),
    "account_age_days": account_age_days,
    "transactions_last_24h": transactions_last_24h,
    "device_changes_30d": device_changes_30d,
    "failed_attempts": failed_attempts,
    "international": international,
    "new_device": new_device,
    "transaction_hour": transaction_hour,
    "is_fraud": is_fraud
})

# Save dataset
df.to_csv("transactions.csv", index=False)

print("Dataset created successfully!")
print(f"Total transactions: {len(df):,}")
print(f"Fraudulent transactions: {df['is_fraud'].sum():,}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")

display(df.head(10))

Dataset created successfully!
Total transactions: 10,000
Fraudulent transactions: 1,430
Fraud rate: 14.30%


,amount,account_age_days,transactions_last_24h,device_changes_30d,failed_attempts,international,new_device,transaction_hour,is_fraud
0,1802.11,1898,2,1,0,1,0,15,0
1,955.02,1656,3,0,2,1,0,2,1
2,2095.80,1873,3,0,2,0,0,0,0
3,5029.27,1055,3,0,2,0,1,10,0
4,867.70,693,6,2,3,0,0,2,0
5,867.71,1698,1,0,0,0,1,15,0
6,5319.92,128,4,0,0,0,0,2,0
7,2362.40,767,4,0,3,0,0,0,1
8,685.76,1318,5,1,2,0,0,7,0
9,1886.65,441,0,0,1,0,0,18,0


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

# Load our dataset
df = pd.read_csv("transactions.csv")

# Features and target
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create the Random Forest model
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

# Train the model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

# Evaluate
print("PAYGUARD MODEL RESULTS")
print("=" * 40)
print(f"Test samples: {len(X_test):,}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probability):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Save the trained model
joblib.dump(model, "payguard_model.pkl")

print("\nModel saved successfully as payguard_model.pkl")

PAYGUARD MODEL RESULTS
Test samples: 2,000
ROC-AUC Score: 0.5552

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.90      0.89      1714
           1       0.25      0.19      0.21       286

    accuracy                           0.80      2000
   macro avg       0.56      0.55      0.55      2000
weighted avg       0.78      0.80      0.79      2000


Model saved successfully as payguard_model.pkl


In [5]:
# Create a stronger, more learnable fraud dataset

np.random.seed(42)

n = 15000

amount = np.random.lognormal(7.0, 1.0, n)
amount = np.clip(amount, 50, 100000)

account_age_days = np.random.randint(1, 2000, n)
transactions_last_24h = np.clip(np.random.poisson(3, n), 0, 30)
device_changes_30d = np.clip(np.random.poisson(1, n), 0, 10)
failed_attempts = np.clip(np.random.poisson(0.8, n), 0, 10)

international = np.random.binomial(1, 0.18, n)
new_device = np.random.binomial(1, 0.20, n)
transaction_hour = np.random.randint(0, 24, n)

# Stronger underlying risk signal
risk = (
    0.000035 * amount
    + 0.10 * transactions_last_24h
    + 0.45 * device_changes_30d
    + 0.55 * failed_attempts
    + 0.9 * international
    + 1.2 * new_device
    + 1.3 * ((transaction_hour < 5) | (transaction_hour > 23))
    + 1.4 * (account_age_days < 30)
)

# Convert risk to probability
probability = 1 / (1 + np.exp(-(risk - 3.5)))

# Generate fraud labels
is_fraud = np.random.binomial(1, probability)

df = pd.DataFrame({
    "amount": amount.round(2),
    "account_age_days": account_age_days,
    "transactions_last_24h": transactions_last_24h,
    "device_changes_30d": device_changes_30d,
    "failed_attempts": failed_attempts,
    "international": international,
    "new_device": new_device,
    "transaction_hour": transaction_hour,
    "is_fraud": is_fraud
})

df.to_csv("transactions.csv", index=False)

print("Improved dataset created!")
print(f"Transactions: {len(df):,}")
print(f"Fraudulent: {df['is_fraud'].sum():,}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")

Improved dataset created!
Transactions: 15,000
Fraudulent: 3,260
Fraud rate: 21.73%


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib

# Load improved dataset
df = pd.read_csv("transactions.csv")

# Separate features and target
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Results
auc = roc_auc_score(y_test, y_prob)

print("=" * 50)
print("       PAYGUARD AI RISK MODEL")
print("=" * 50)
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")
print(f"ROC-AUC Score    : {auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Save model
joblib.dump(model, "payguard_model.pkl")

print("Model saved successfully!")

       PAYGUARD AI RISK MODEL
Training samples : 12,000
Testing samples  : 3,000
ROC-AUC Score    : 0.7519

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      2348
           1       0.48      0.52      0.50       652

    accuracy                           0.77      3000
   macro avg       0.67      0.68      0.68      3000
weighted avg       0.78      0.77      0.78      3000

Model saved successfully!


In [8]:
import gradio as gr
import pandas as pd
import numpy as np
import joblib

# Load trained model
model = joblib.load("payguard_model.pkl")

def assess_transaction(
    amount,
    account_age_days,
    transactions_last_24h,
    device_changes_30d,
    failed_attempts,
    international,
    new_device,
    transaction_hour
):
    # Create input in the exact same order used during training
    data = pd.DataFrame([{
        "amount": amount,
        "account_age_days": account_age_days,
        "transactions_last_24h": transactions_last_24h,
        "device_changes_30d": device_changes_30d,
        "failed_attempts": failed_attempts,
        "international": international,
        "new_device": new_device,
        "transaction_hour": transaction_hour
    }])

    # Get fraud probability
    probability = model.predict_proba(data)[0][1]

    # Convert to risk score
    risk_score = round(probability * 100)

    # Determine risk level
    if risk_score >= 60:
        level = "🔴 HIGH RISK"
        recommendation = "Block or send this transaction for manual review."
    elif risk_score >= 40:
        level = "🟠 MEDIUM RISK"
        recommendation = "Request additional verification before approving."
    else:
        level = "🟢 LOW RISK"
        recommendation = "Transaction appears relatively safe to approve."

    # Generate human-readable explanations
    reasons = []

    if amount > 50000:
        reasons.append("Unusually high transaction amount")
    if transactions_last_24h > 8:
        reasons.append("High transaction frequency in the last 24 hours")
    if device_changes_30d > 3:
        reasons.append("Multiple device changes recently")
    if failed_attempts > 2:
        reasons.append("Multiple failed payment attempts")
    if international == 1:
        reasons.append("International transaction")
    if new_device == 1:
        reasons.append("Transaction from a new device")
    if transaction_hour < 5 or transaction_hour > 23:
        reasons.append("Transaction occurred during unusual hours")
    if account_age_days < 30:
        reasons.append("Recently created account")

    if not reasons:
        reasons.append("No major behavioral risk signals detected")

    reason_text = "\n".join([f"• {r}" for r in reasons])

    result = f"""
# {level}

### Risk Score: {risk_score}/100

**AI Assessment**

{recommendation}

### Risk Signals

{reason_text}

### Model Probability
Estimated probability of fraud: **{probability:.1%}**
"""

    return result


# Interface
with gr.Blocks(title="PayGuard AI Risk Manager") as demo:

    gr.Markdown("""
# 🛡️ PayGuard
## AI-Powered Transaction Risk Manager

Analyze payment transactions using machine learning and identify
potentially fraudulent behavior before approval.
""")

    gr.Markdown("### Enter Transaction Details")

    with gr.Row():
        with gr.Column():
            amount = gr.Number(
                label="Transaction Amount (₹)",
                value=5000,
                minimum=50,
                maximum=100000
            )

            account_age_days = gr.Number(
                label="Account Age (days)",
                value=365,
                minimum=1,
                maximum=2000
            )

            transactions_last_24h = gr.Number(
                label="Transactions in Last 24 Hours",
                value=3,
                minimum=0,
                maximum=30
            )

            device_changes_30d = gr.Number(
                label="Device Changes in Last 30 Days",
                value=1,
                minimum=0,
                maximum=10
            )

        with gr.Column():
            failed_attempts = gr.Number(
                label="Failed Payment Attempts",
                value=0,
                minimum=0,
                maximum=10
            )

            international = gr.Radio(
                [0, 1],
                label="International Transaction?",
                value=0
            )

            new_device = gr.Radio(
                [0, 1],
                label="New Device?",
                value=0
            )

            transaction_hour = gr.Slider(
                minimum=0,
                maximum=23,
                step=1,
                value=14,
                label="Transaction Hour (0–23)"
            )

    analyze_button = gr.Button(
        "🔍 Analyze Transaction",
        variant="primary"
    )

    output = gr.Markdown()

    analyze_button.click(
        fn=assess_transaction,
        inputs=[
            amount,
            account_age_days,
            transactions_last_24h,
            device_changes_30d,
            failed_attempts,
            international,
            new_device,
            transaction_hour
        ],
        outputs=output
    )

    gr.Markdown("""
---
### How PayGuard Works

**Transaction Data → ML Model → Fraud Probability → Risk Score → Actionable Explanation**

The underlying model is a Random Forest classifier trained on simulated
payment-risk data for this prototype.
""")

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92f9d757a8f68c08a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
import os

print("PROJECT FILES")
print("=" * 40)

for file in os.listdir("."):
    if file.endswith((".py", ".csv", ".pkl", ".md", ".txt")):
        print(file)

PROJECT FILES
payguard_model.pkl
transactions.csv
